In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool


In [2]:
@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"


@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

In [5]:
checkpointer = InMemorySaver()

In [ ]:
hitl_agent = create_agent(
    model="mistral-small-2506",
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,
                "delete_records": True,
            }
        )
    ],
    checkpointer=checkpointer,
)

In [ ]:
# Step 1: Invoke - agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
{"messages": [
    {
        "role": "user", 
        "content": "Send an email to team@company.com about the Q4 results"
        }
    ]},
    config=config
)

print(" === Agent paused - awaiting human approval === ")
print(result)

 === Agent paused - awaiting human approval === 
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='3b48303d-3a20-471d-b57e-26dd6c968f80'), AIMessage(content='Sure, what would you like the subject and body of the email to be?', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 221, 'total_tokens': 238, 'completion_tokens': 17, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-2506', 'model': 'mistral-small-2506', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019f7966-5077-75f3-8361-f58934383dc9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 221, 'output_tokens': 17, 'total_tokens': 238}), HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='22daafe7-2707-4ad9-b12d-416549bbc02a'), AIMessage(content='Sure, please provide the su

In [10]:
# Step 1: Invoke - agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
{"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
config=config

)

print(" === Agent paused - awaiting human approval === ")
print(result)

 === Agent paused - awaiting human approval === 
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='3b48303d-3a20-471d-b57e-26dd6c968f80'), AIMessage(content='Sure, what would you like the subject and body of the email to be?', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 221, 'total_tokens': 238, 'completion_tokens': 17, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-2506', 'model': 'mistral-small-2506', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019f7966-5077-75f3-8361-f58934383dc9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 221, 'output_tokens': 17, 'total_tokens': 238}), HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='22daafe7-2707-4ad9-b12d-416549bbc02a'), AIMessage(content='Sure, please provide the su

In [11]:
# Step 2: Human reviews and APPROVE
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type" : "approve"}]}),
    config=config     # Same thread_id resumes the paused session
)

print(" === Approved! Final response === ")
print(approved_result["messages"][-1].content)

 === Approved! Final response === 
Here is a draft for the email:

---

**Subject:** Q4 Results Overview

**Body:**

Dear Team,

I hope this message finds you well.

I am pleased to share the Q4 results with you. We have made significant progress this quarter, and I would like to highlight some key achievements and insights.

1. **Revenue Growth:**
   - Our revenue increased by 15% compared to Q3, reaching $5 million.

2. **Customer Satisfaction:**
   - Customer satisfaction scores improved by 20%, reflecting our commitment to quality and service.

3. **Operational Efficiency:**
   - We streamlined our operations, reducing costs by 10%.

4. **Future Goals:**
   - We aim to build on these successes in Q1 by focusing on new market opportunities and further improving our customer experience.

Thank you for your hard work and dedication. Your contributions are invaluable to our continued success.

Best regards,

[Your Name]
[Your Position]
[Your Contact Information]

---


In [12]:
# Step 3: Alternative - Human REJECTS
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2

)

rejected_result = hitl_agent. invoke(
    Command(resume={"decisions": [{"type" : "reject","peason": "Too risky, needs DBA review"}]}),
    config=config2
)

print(" === Rejected! Final response === ")
print(rejected_result["messages"][-1].content)

 === Rejected! Final response === 
To delete all records from the users table where active is false, you can use the following SQL command:

```sql
DELETE FROM users WHERE active = false;
```

However, if you want me to execute this command, please let me know.
